# Run ARC SFT Strategy Enrichment & SFT Dataset Extraction

Clone the ARC repository from GitLab, install dependencies, run the SFT strategy enricher (with structured JSON output: `strategy`, `response`, `justification`), and extract the conversational SFT dataset (`sft_execution_data.jsonl`).

## 1. Runtime Parameters

Edit these values before running the notebook if needed. For best performance, use a GPU runtime (T4, V100, A100).

In [ ]:
from pathlib import Path

REPO_URL = "https://gitlab.com/beryl.hoe/arc.git"
PROJECT_DIR = Path("/content/arc")
INPUT_PATH = PROJECT_DIR / "data" / "sft_train.jsonl"
OUTPUT_PATH = PROJECT_DIR / "data" / "enriched_sft_train.jsonl"
SFT_EXECUTION_PATH = PROJECT_DIR / "data" / "sft_execution_data.jsonl"
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_NEW_TOKENS = 256
MAX_INPUT_TOKENS = 4096
MAX_DOCUMENT_CHARS = 1200
MAX_TOTAL_DOCUMENT_CHARS = 6000
QUESTION_BATCH_SIZE = 2
SAVE_EXECUTIONS = True
OVERWRITE = True
FORCE_RECLONE = False

## 2. Install System Packages

In [ ]:
!apt-get -qq update
!apt-get -qq install -y git git-lfs wget > /dev/null
!git lfs install

## 3. Clone the Repository

In [ ]:
import shutil
import subprocess

if FORCE_RECLONE and PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if PROJECT_DIR.exists():
    print(f"Repository already exists at {PROJECT_DIR}; pulling latest changes.")
    subprocess.run(["git", "pull"], cwd=PROJECT_DIR, check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

print(f"Project directory: {PROJECT_DIR}")

## 4. Install Python Dependencies

In [ ]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True,
)

## 5. Preflight Check

In [ ]:
sys.path.insert(0, str(PROJECT_DIR))

preflight = subprocess.run(
    [
        sys.executable,
        "-c",
        "from src.enrich_sft_with_strategies import main; from src.extract_sft_from_executions import main as extract_main; print('python ok')",
    ],
    cwd=PROJECT_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
print("--- STDOUT ---")
print(preflight.stdout)
print("--- STDERR ---")
print(preflight.stderr)
if preflight.returncode != 0:
    raise RuntimeError(f"Preflight failed with exit code {preflight.returncode}")

## 6. Run the Strategy Enricher

Executes the 8 ICR strategies with structured JSON prompts requesting `strategy`, `response`, and `justification` fields.

In [ ]:
cmd = [
    sys.executable,
    "-u",
    "-m",
    "src.enrich_sft_with_strategies",
    "--input-path",
    str(INPUT_PATH),
    "--output-path",
    str(OUTPUT_PATH),
    "--model-name",
    MODEL_NAME,
    "--max-new-tokens",
    str(MAX_NEW_TOKENS),
    "--max-input-tokens",
    str(MAX_INPUT_TOKENS),
    "--max-document-chars",
    str(MAX_DOCUMENT_CHARS),
    "--max-total-document-chars",
    str(MAX_TOTAL_DOCUMENT_CHARS),
    "--question-batch-size",
    str(QUESTION_BATCH_SIZE),
]

if SAVE_EXECUTIONS:
    cmd.append("--save-executions")
if OVERWRITE:
    cmd.append("--overwrite")

print("Running:", " ".join(cmd))
process = subprocess.Popen(
    cmd,
    cwd=PROJECT_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="")
returncode = process.wait()
if returncode != 0:
    raise RuntimeError(f"Strategy enricher failed with exit code {returncode}")

## 7. Inspect Enricher Output

In [ ]:
import json
from collections import Counter

if OUTPUT_PATH.exists():
    records = []
    with OUTPUT_PATH.open('r', encoding='utf-8') as handle:
        for line in handle:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    counter = Counter(len(record.get('valid_strategies', [])) for record in records)
    print(f"Total questions processed: {len(records)}")
    print('Distribution of valid strategies:')
    for key in sorted(counter):
        print(f'  {key} strategies: {counter[key]}')

    print('\nSample questions with valid strategies & JSON execution responses:')
    count = 0
    for record in records:
        valid = record.get('valid_strategies', [])
        if valid:
            sid = valid[0]
            exec_resp = record.get('executions', {}).get(sid, {}).get('response', '')
            print(f"\n  query_id: {record.get('query_id', '')}")
            print(f'  valid_strategies: {valid}')
            print(f'  sample execution response ({sid}):\n{exec_resp[:300]}')
            count += 1
            if count >= 2:
                break
else:
    print(f"{OUTPUT_PATH.name}: missing")

## 8. Extract Structured SFT Dataset

Extracts conversational training examples (`messages` format) into `sft_execution_data.jsonl` from each valid strategy execution.

In [ ]:
extract_cmd = [
    sys.executable,
    "-u",
    "-m",
    "src.extract_sft_from_executions",
]

print("Running:", " ".join(extract_cmd))
process = subprocess.Popen(
    extract_cmd,
    cwd=PROJECT_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="")
returncode = process.wait()
if returncode != 0:
    raise RuntimeError(f"SFT extraction failed with exit code {returncode}")

## 9. Inspect Extracted SFT Dataset (Strict JSON Messages)

Verifies line count and checks that the assistant response is strict JSON with `strategy`, `response`, and `justification`.

In [ ]:
import json

if SFT_EXECUTION_PATH.exists():
    with SFT_EXECUTION_PATH.open('r', encoding='utf-8') as handle:
        lines = [line.strip() for line in handle if line.strip()]
    print(f"Total SFT examples generated: {len(lines)}")
    if lines:
        first_example = json.loads(lines[0])
        print("\n--- First SFT Example ---")
        for msg in first_example.get("messages", []):
            role = msg.get("role", "")
            content = msg.get("content", "")
            print(f"\n[{role.upper()}]:")
            print(content[:400] + ("..." if len(content) > 400 else ""))

        # Verify assistant content is parseable JSON
        assistant_content = first_example["messages"][2]["content"]
        try:
            parsed = json.loads(assistant_content)
            print("\n Assistant message is valid JSON:")
            print(json.dumps(parsed, indent=2))
        except Exception as e:
            print(f"\n Assistant content is not valid JSON: {e}")
else:
    print(f"{SFT_EXECUTION_PATH.name}: missing")